Recommendation systems are intelligent systems used to suggest relevant items to users based on their preferences and behavior. These systems are widely used in platforms such as Netflix, Amazon, YouTube, and Spotify to improve user experience by providing personalized recommendations.

In this assignment, a content-based recommendation system is implemented using cosine similarity to recommend similar anime. The system analyzes features such as genre, rating, number of episodes, and popularity to determine how similar two anime are.

The goal of this system is to recommend anime that share similar characteristics with a given anime.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import CountVectorizer

2. Data Description

The dataset used in this project contains information about various anime. Each record represents an anime along with its attributes.

In [3]:
df = pd.read_csv("/content/anime (1).csv")

df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [5]:
df.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [6]:
df['genre'] = df['genre'].fillna('')
df['rating'] = df['rating'].fillna(df['rating'].mean())
df['episodes'] = df['episodes'].replace('Unknown',0)
df['episodes'] = df['episodes'].astype(int)

3. Data Preprocessing

Data preprocessing is an important step in building machine learning models. It involves cleaning and preparing the dataset so that it can be effectively used for analysis and model building.

In [7]:
df.sort_values(by='rating', ascending=False).head(10)

,anime_id,name,genre,type,episodes,rating,members
10464,33662,Taka no Tsume 8: Yoshida-kun no X-Files,"Comedy, Parody",Movie,1,10.00,13
10400,30120,Spoon-hime no Swing Kitchen,"Adventure, Kids",TV,0,9.60,47
9595,23005,Mogura no Motoro,Slice of Life,Movie,1,9.50,62
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
9078,33607,Kahei no Umi,Historical,Movie,1,9.33,44
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
10786,26313,Yakusoku: Africa Mizu to Midori,"Drama, Kids",OVA,1,9.25,53
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [8]:
df.sort_values(by='members', ascending=False).head(10)

,anime_id,name,genre,type,episodes,rating,members
40,1535,Death Note,"Mystery, Police, Psychological, Supernatural, ...",TV,37,8.71,1013917
86,16498,Shingeki no Kyojin,"Action, Drama, Fantasy, Shounen, Super Power",TV,25,8.54,896229
804,11757,Sword Art Online,"Action, Adventure, Fantasy, Game, Romance",TV,25,7.83,893100
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
159,6547,Angel Beats!,"Action, Comedy, Drama, School, Supernatural",TV,13,8.39,717796
19,1575,Code Geass: Hangyaku no Lelouch,"Action, Mecha, Military, School, Sci-Fi, Super...",TV,25,8.83,715151
841,20,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
445,10620,Mirai Nikki (TV),"Action, Mystery, Psychological, Shounen, Super...",TV,26,8.07,657190
131,4224,Toradora!,"Comedy, Romance, School, Slice of Life",TV,25,8.45,633817


4. Feature Extraction
Feature extraction involves selecting and transforming relevant data attributes into numerical representations that can be used by machine learning algorithms.

The following features were selected:

Genre

Rating

Number of Episodes

Number of Members

In [9]:
vectorizer = CountVectorizer(tokenizer=lambda x: x.split(","))
genre_matrix = vectorizer.fit_transform(df['genre'])

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [10]:
scaler = MinMaxScaler()

numeric_features = df[['rating','episodes','members']]

scaled_features = scaler.fit_transform(numeric_features)

In [11]:
from scipy.sparse import hstack

feature_matrix = hstack([genre_matrix, scaled_features])

5. Cosine Similarity

Cosine similarity is used to measure the similarity between two vectors by calculating the cosine of the angle between them.

The value of cosine similarity ranges between 0 and 1:

1 indicates that two items are highly similar.

0 indicates no similarity.

In this project, cosine similarity is used to compare the feature vectors of different anime and determine how similar they are to each other.

In [12]:
cosine_sim = cosine_similarity(feature_matrix)

6. Recommendation System

A recommendation function was created to suggest anime similar to a given anime title.

In [13]:
def recommend_anime(anime_name, num_recommendations=5):

    # Get anime index
    idx = df[df['name'] == anime_name].index[0]

    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Skip first (itself)
    sim_scores = sim_scores[1:num_recommendations+1]

    anime_indices = [i[0] for i in sim_scores]

    return df['name'].iloc[anime_indices]

In [14]:
recommend_anime("Steins;Gate",10)

,name
59,Steins;Gate Movie: Fuka Ryouiki no Déjà vu
126,Steins;Gate: Oukoubakko no Poriomania
196,Steins;Gate: Kyoukaimenjou no Missing Link - D...
10898,Steins;Gate 0
5283,Final Fantasy: The Spirits Within
1578,Sakasama no Patema: Beginning of the Day
1594,Mai-Otome 0: S.ifr
9091,Kaitei Toshi no Dekiru made
10414,Subarashii Sekai Ryokou: New York Tabi &quot;C...
3492,Glass no Hana to Kowasu Sekai


In [15]:
def recommend_with_threshold(anime_name, threshold=0.5):

    idx = df[df['name'] == anime_name].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))

    recommendations = [i for i in sim_scores if i[1] > threshold]

    recommendations = sorted(recommendations, key=lambda x: x[1], reverse=True)[1:10]

    anime_indices = [i[0] for i in recommendations]

    return df['name'].iloc[anime_indices]

8. Possible Improvements

The system can be improved using the following methods:

Using TF-IDF vectorization instead of CountVectorizer.

Adding user rating datasets for collaborative filtering.

Implementing hybrid recommendation systems.

Applying deep learning recommendation models.

These improvements can enhance recommendation accuracy and personalization.

Conclusion

In this assignment, a content-based anime recommendation system was implemented using cosine similarity. The dataset was preprocessed, relevant features were extracted, and similarity scores were computed to recommend similar anime titles. This approach demonstrates how machine learning techniques can be applied to build simple yet effective recommendation systems.